# Phase 40 — learning run: held-out generalisation of NTK-Mirror gates (Kaggle)

A **single honest worker** trains the K=5000 gate vector on the **deployed** Cloudflare
coordinator and measures **held-out** loss every round. A monotone drop on problems the
gates were *never trained on* is genuine generalisation — the answer to "is the learning
real, or just memorising a tiny batch?"

This notebook is a **measurement, not a guaranteed win.** K=5000 is ~0.001% of
Qwen2.5-0.5B's parameters — small capacity. Honest outcomes are *drops modestly*, *flat*,
or *overfits (train down, eval not)*. All three are reportable; the notebook prints the
verdict either way.

## Before clicking ▶ Run All
1. Right sidebar (gear) → **Accelerator** → `GPU T4` (any GPU).
2. **Internet** → `On` (pip + git + reaching the Worker).
3. Run cells top to bottom.

~15–25 min on T4 for R=300. Hits `https://postnet-cf.abgunaydin94.workers.dev` and **resets
its NTK state first**. Outputs land in `/kaggle/working/learn/`:
`trajectory.csv`, `descent.png`, `RESULT.md`.


## Cell 1 — install deps + clone repos (pulls the updated verifier)

In [ ]:
import subprocess, os, sys, urllib.request

try:
    urllib.request.urlopen("https://github.com", timeout=5).close()
    print("✓ internet reachable")
except Exception as e:
    sys.exit(f"✗ INTERNET DISABLED: {e}\n  Right sidebar (gear) → Internet → On, then re-run.")

print("→ pip install …")
subprocess.run(["pip", "-q", "install", "transformers", "torch", "numpy", "requests", "matplotlib"], check=True)

os.makedirs("/kaggle/working", exist_ok=True)
for name, url in [("ntkmirror", "https://github.com/leochlon/ntkmirror.git"),
                  ("postnet-cf", "https://github.com/abgnydn/postnet-cf.git")]:
    dst = f"/kaggle/working/{name}"
    if os.path.isdir(dst):
        print(f"→ git pull {name}")
        subprocess.run(["git", "-C", dst, "pull", "--ff-only"], check=True)
    else:
        print(f"→ git clone {name}")
        subprocess.run(["git", "clone", url, dst], check=True)

print("→ pip install -e ntkmirror")
subprocess.run(["pip", "-q", "install", "-e", "/kaggle/working/ntkmirror"], check=True)
print("OK")

## Cell 2 — build a held-out corpus (32 train / 32 eval, disjoint)

Same 2-digit-addition carry format the gates were selected on, so there is **no task-shift
confound** — just unseen operand pairs. Train and eval pairs are disjoint by construction,
so eval loss measures generalisation, not memorisation.

In [ ]:
import json, random, os
os.makedirs("/kaggle/working/learn", exist_ok=True)

def solve(a, b):
    o = (a % 10) + (b % 10)
    t = (a // 10) + (b // 10) + (o // 10)
    return (f" Add ones: {a%10}+{b%10}={o}, write {o%10} carry {o//10}. "
            f"Tens: {a//10}+{b//10}+{o//10}={t}. Answer: {a+b}")

rng = random.Random(40)
pairs = set()
while len(pairs) < 64:
    a, b = rng.randint(10, 89), rng.randint(10, 89)
    if a + b < 100:
        pairs.add((a, b))
pairs = list(pairs); rng.shuffle(pairs)
train, evl = pairs[:32], pairs[32:64]
assert not (set(train) & set(evl)), "train/eval overlap!"

def write(path, ps):
    with open(path, "w") as f:
        for a, b in ps:
            f.write(json.dumps({"prompt": f"Problem: {a} + {b} = ?\nSolution:",
                                "completion": solve(a, b)}) + "\n")

write("/kaggle/working/learn/train.jsonl", train)
write("/kaggle/working/learn/eval.jsonl", evl)
print(f"train={len(train)}  eval={len(evl)}  disjoint ✓")
a, b = train[0]
print("sample:", json.dumps({"prompt": f"Problem: {a} + {b} = ?\\nSolution:", "completion": solve(a, b)}))

## Cell 3 — GPU check + reset the deployed coordinator

In [ ]:
import torch, requests
assert torch.cuda.is_available(), "No GPU. Right sidebar → Accelerator → GPU T4."
print("✓ GPU:", torch.cuda.get_device_name(0))

COORD = "https://postnet-cf.abgunaydin94.workers.dev"
_UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
       "(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36 postnet-ntk/1.0")
_S = requests.Session(); _S.headers.update({"User-Agent": _UA})

print("reset:", _S.post(f"{COORD}/api/ntk/reset").json())
s = _S.get(f"{COORD}/api/ntk/state").json()
print(f"R={s['round']}  eta={s['eta']}  K={s.get('K')}  pending_audits={s.get('pending_audits')}")

## Cell 4 — the learning run (single honest worker, R=300)

Held-out eval loss is computed at the current θ every round (never optimised against) and
written to `trajectory.csv`.

**Single-worker regime (honest caveat).** The deployed coordinator needs `TARGET_PROPOSALS=2`
before it advances a round, so one worker advances every *two* submissions — ~150 gate
updates for 300 loop iterations. Also, the no-self-audit rule means a lone worker never
closes its own audits, so η stays fixed at its init (1e-3) — sym-AIMD does not fire (the
documented single-honest degenerate regime). Descent still happens: a proposal is applied
whenever its Δ<0, independent of audits. A full-rate, η-adapting run needs ≥2 honest workers
(natural follow-up: pin one per GPU on a T4×2 kernel).

In [ ]:
import subprocess, time, os
log = "/kaggle/working/learn/run.log"
cmd = [
    "python", "/kaggle/working/postnet-cf/scripts/ntk-verifier.py",
    "--coord", COORD,
    "--model", "Qwen/Qwen2.5-0.5B-Instruct",
    "--train",    "/kaggle/working/learn/train.jsonl",
    "--eval",     "/kaggle/working/learn/eval.jsonl",
    "--artifact", "/kaggle/working/postnet-cf/public/data/qwen05b-math-gates-k5000.bin",
    "--rounds", "300", "--trials", "4",
    "--batch-size", "32", "--eval-batch-size", "32", "--max-length", "64",
    "--device", "cuda", "--dtype", "fp32",
    "--seed", "40", "--worker-id", "kaggle-learn",
    "--trajectory", "/kaggle/working/learn/trajectory.csv",
    "--reset",
]
t0 = time.time()
with open(log, "w") as f:
    rc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
print(f"exit={rc.returncode}   wall={time.time()-t0:.0f}s")
print("\n".join(open(log).read().splitlines()[-30:]))

## Cell 5 — plot train vs held-out descent + write RESULT.md

In [ ]:
import csv, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

rounds, tr, ev = [], [], []
with open("/kaggle/working/learn/trajectory.csv") as f:
    for row in csv.DictReader(f):
        rounds.append(int(row["round"]))
        tr.append(float(row["train_loss"]))
        ev.append(float(row["eval_loss"]) if row["eval_loss"] else None)

plt.figure(figsize=(8, 5))
plt.plot(rounds, tr, label="train (32 problems)", lw=1.5)
er = [r for r, e in zip(rounds, ev) if e is not None]
ee = [e for e in ev if e is not None]
if ee:
    plt.plot(er, ee, label="held-out eval (32 unseen)", lw=1.5)
plt.xlabel("round"); plt.ylabel("cross-entropy loss")
plt.title("NTK-Mirror gate training — train vs held-out (K=5000, Qwen-0.5B)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("/kaggle/working/learn/descent.png", dpi=120)
print("saved descent.png")

def ends(xs):
    xs = [x for x in xs if x is not None]
    return (xs[0], xs[-1], xs[-1] - xs[0]) if xs else (None, None, None)

t0v, t1v, td = ends(tr)
e0, e1, ed = ends(ev)
if ed is None:
    verdict = "no held-out eval recorded"
elif ed < -1e-4:
    verdict = "held-out loss DROPPED — genuine generalisation"
elif abs(ed) <= 1e-4:
    verdict = "held-out loss FLAT — no measurable generalisation"
else:
    verdict = "held-out loss ROSE — train-only fit / overfitting"

def f4(x): return f"{x:.4f}" if x is not None else "—"
def fdelta(x): return f"{x:+.4f}" if x is not None else "—"

md_txt = f"""# Phase 40 learning run — result

- **Corpus:** 32 train / 32 held-out 2-digit-addition problems (disjoint, carry format).
- **Gates:** K=5000 on frozen Qwen2.5-0.5B-Instruct, SPSA tournament on the deployed coordinator.
- **Rounds:** {len(rounds)}.

| metric | start | final | Δ |
|---|---|---|---|
| train loss | {f4(t0v)} | {f4(t1v)} | {fdelta(td)} |
| held-out eval loss | {f4(e0)} | {f4(e1)} | {fdelta(ed)} |

**Verdict:** {verdict}.

![descent](descent.png)
"""
open("/kaggle/working/learn/RESULT.md", "w").write(md_txt)
print(md_txt)

## Cell 6 — outputs

Everything in `/kaggle/working/learn/` is downloadable from the right-sidebar **Output**
panel after the notebook saves.

In [ ]:
import os
print(os.listdir("/kaggle/working/learn"))